# 06 · Entrenar SmolVLA con nuestros datos

El modelo base no sabe mover un G1: se preentrenó con brazos de 6
articulaciones (`learn/l7-1`). Acá lo entrenamos con nuestras grabaciones. Se
entrena sólo el experto de acciones y las proyecciones (100 M de 450 M); el
VLM queda congelado (`learn/l7-7`).

Mismos episodios que la primera ACT: 54 para entrenar y 7 para evaluar
(`notes/act-split.json`), así la comparación de `07` es de igual a igual.

In [ ]:
import json
import re
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR

plt.rcParams["figure.dpi"] = 110
RUN = CHECKPOINTS_DIR / "smolvla_g1_v1"
LOG = CHECKPOINTS_DIR / "smolvla_g1_v1.log"

## 1. Tres decisiones antes de entrenar

**a. Una copia de `smolvla_base` con las formas de nuestro robot.** En LeRobot
0.4.1, al partir de un modelo preentrenado, la configuración conserva las
formas del robot original en vez de tomarlas del dataset. Habría generado 6
articulaciones de nuestras 26. La versión actual ya lo corrige; mientras
tanto, `toolkit/policy/smolvla_g1_base.py` reescribe sólo esas formas en una
copia local (`python -m toolkit.policy.smolvla_g1_base`):

In [ ]:
from huggingface_hub import hf_hub_download

base = json.loads(open(hf_hub_download("lerobot/smolvla_base", "config.json")).read())
ours = json.loads((CHECKPOINTS_DIR / "smolvla_base_g1" / "config.json").read_text())
for key in ["input_features", "output_features"]:
    print(f"{key}\n  smolvla_base: {base[key]}\n  nuestra copia: {ours[key]}")
changed = [k for k in base if base[k] != ours.get(k)]
print("claves que cambian:", changed)

**b. Entrenamiento y evaluación como datasets separados.** En 0.4.1, pedir
"entrená sólo con estos episodios" no filtra nada en un dataset local
(`toolkit/README.md`). Partimos el dataset en dos con
`python -m toolkit.dataset.split_act_v1`. De paso, las estadísticas de
normalización salen sólo de los 54 de entrenamiento: sin la fuga que tuvo la
primera ACT (`notes/act-first-model.md`).

In [ ]:
for name in ["train", "heldout"]:
    info = json.loads((DATASETS_DIR / "local" / f"pick_mameluco_act_v1_{name}" / "meta" / "info.json").read_text())
    print(f"{name:>8}: {info['total_episodes']} episodios · {info['total_frames']} cuadros")

**c. El script oficial, sin tocar.** Con esas dos cosas resueltas afuera, el
entrenamiento es el `lerobot-train` de LeRobot con la receta de la
documentación de SmolVLA. Las cámaras se renombran a `camera1`, la convención
de SmolVLA para la vista de arriba. Cambiamos el tamaño de lote a 32 para que
entre cómodo en la GPU de la workstation (la documentación usa 64).

## 2. El comando

```bash
lerobot-train \
  --policy.path=data/checkpoints/smolvla_base_g1 \
  --dataset.repo_id=local/pick_mameluco_act_v1_train \
  --dataset.root=data/datasets/local/pick_mameluco_act_v1_train \
  --rename_map='{"observation.images.cam_right_high": "observation.images.camera1"}' \
  --batch_size=32 --steps=20000 --save_freq=5000 --log_freq=100 \
  --output_dir=data/checkpoints/smolvla_g1_v1 --job_name=smolvla_g1_v1 \
  --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false \
  > data/checkpoints/smolvla_g1_v1.log 2>&1
```

## 3. Cómo fue

Leemos el log: una línea cada 100 pasos.

In [ ]:
pattern = re.compile(r"INFO (\S+ \S+) .*?step:\S+ .*?loss:([\d.]+) grdn:([\d.]+) lr:([\d.e+-]+) updt_s:([\d.]+) data_s:([\d.]+)")
rows = [pattern.search(line) for line in LOG.read_text().splitlines()]
rows = [r for r in rows if r]
step = 100 * np.arange(1, len(rows) + 1)  # el log redondea los pasos ("5K"): los contamos
when = [datetime.strptime(r.group(1), "%Y-%m-%d %H:%M:%S") for r in rows]
loss, grad, lr, update_s, data_s = (np.array([float(r.group(i)) for r in rows]) for i in range(2, 7))

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(step, loss)
axes[0].set_yscale("log")
axes[0].set_title("pérdida (escala logarítmica)")
axes[1].plot(step, lr)
axes[1].set_title("tasa de aprendizaje")
axes[2].plot(step, grad)
axes[2].set_title("largo del gradiente")
for ax in axes:
    ax.set_xlabel("paso")
plt.tight_layout()
hours = (when[-1] - when[0]).total_seconds() / 3600
print(f"{step[-1]} pasos en {hours:.1f} h · {update_s.mean():.2f} s por paso de cálculo + {data_s.mean():.3f} s de datos")
print(f"pérdida: primeros 500 pasos {loss[:5].mean():.3f} → últimos 1000 {loss[-10:].mean():.4f}")

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## 4. Qué quedó

Un punto de control cada 5000 pasos. `last` apunta al último: es el que usan
`learn/l7-5` a `l7-7` y `07`.

In [ ]:
for checkpoint in sorted((RUN / "checkpoints").iterdir()):
    model = checkpoint / "pretrained_model" / "model.safetensors"
    if model.exists():
        print(f"  {checkpoint.name:>8}: {model.stat().st_size / 2**20:.0f} MB")

## Lo que sigue

- `07`: medirlo en los 7 episodios que no vio, con la misma vara que la
  primera ACT.
- Experimento de una variable: arreglar el problema del relleno de fin de
  episodio (`learn/l7-7`, problema 1) y comparar contra este entrenamiento, que
  queda como referencia.